In [1]:
import os
import random
import json
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from typing import Dict

In [2]:
class LocalLLM:
    def __init__(self, model_id: str, device: str = 'cuda'):
        self.tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_id,
            device_map='auto',
            torch_dtype=torch.float16,
            trust_remote_code=True
        )
        self.device = self.model.device

    def __call__(self, prompt: str, **generate_kwargs) -> str:
        inputs = self.tokenizer(prompt, return_tensors='pt').to(self.device)
        defaults = dict(max_new_tokens=256, do_sample=False)
        params = {**defaults, **generate_kwargs}
        with torch.no_grad():
            out = self.model.generate(**inputs, **params)
        text = self.tokenizer.decode(out[0], skip_special_tokens=True)
        return text[len(prompt):].strip()

# Cache and model mapping
LLM_CACHE: Dict[str, LocalLLM] = {}
MODEL_MAP = {
    'qwen2.5': 'Qwen/Qwen2.5-7B-Instruct',
    'llama3.1': 'meta-llama/Llama-3.1-8B-Instruct',
    'gemma2': 'google/gemma-2-9b-it'
}


In [3]:

def get_llm(key: str) -> LocalLLM:
    if key not in LLM_CACHE:
        model_id = MODEL_MAP[key]
        LLM_CACHE[key] = LocalLLM(model_id)
    return LLM_CACHE[key]

In [4]:
# Configuration
root_dir = 'experiment_clusters'
templates_dir = './'
output_dir = './output'
model_key = 'llama3.1'  # choose from MODEL_MAP
# model_key = 'qwen2.5'
# model_key = 'gemma2'
group_size = 5

In [5]:

with open("eval_prompts_same_first_global.json", "r") as f:
    eval_prompts = json.load(f)


In [6]:
from pathlib import Path
out_dir = Path("outputs")
out_dir.mkdir(exist_ok=True)

llm = get_llm(model_key)

for itype, scenarios in eval_prompts.items():
    if itype == "json":
        continue
    for scenario, items in scenarios.items():
        # prepare a fresh bucket for this series
        bucket = []

        for entry in items:
            try:
                response = llm(entry['prompt'], temperature=0.0)
                # response = llm(entry['prompt'], temperature=0.0)
                lines = response.splitlines()
                pred_id = lines[0].strip()

                bucket.append({
                    'model':        model_key,
                    'input_type':   itype,
                    'scenario':     scenario,
                    'outlier_id':   entry['outlier_id'],
                    'predicted_id': pred_id,
                    'raw_response': response
                })
            except:
                raise Exception("error processing {}".format(entry))

        # once this (itype, scenario) is done, write it out immediately
        safe_key = f"{llm}_{itype}__{scenario}".replace(" ", "_")
        filepath = out_dir / f"{safe_key}.json"
        with open(filepath, 'w', encoding='utf-8') as f:
            json.dump(bucket, f, indent=2, ensure_ascii=False)

        print(f"Wrote {len(bucket)} records to {filepath!r}")

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

/home/hice1/hzhang931/scratch/planscape/venv/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/hice1/hzhang931/scratch/planscape/venv/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token

Wrote 100 records to PosixPath('outputs/<__main__.LocalLLM_object_at_0x155446e7a6d0>_description__same_first.json')


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for

Wrote 100 records to PosixPath('outputs/<__main__.LocalLLM_object_at_0x155446e7a6d0>_description__same_first_global.json')


In [ ]:
# Save results
with open(os.path.join(output_dir, f'llm_results_{model_key}.json'), 'w') as f:
    json.dump(results, f, indent=2) 
